# Lilly — train the Bosnian to English model

Set these in the panel on the right before running anything:

- **Session options → Accelerator → GPU** (P100 or T4; the notebook uses one either way)
- **Session options → Internet → On**
- **Input → Add Input → Datasets** → your uploaded copy of `models/lilly/translate`.
  Upload it once: Kaggle → Datasets → New Dataset → drag the folder in. The weights are
  too big for git, so this is how they reach the machine.

Then **Save Version → Save & Run All (Commit)** and close the tab. It keeps running
without you, about 2-3 hours, and the result waits in that version's **Output** tab.

Every step below stops the run if it fails, so a green version means it really worked.


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://object.pouta.csc.fi"):
    reachable(host)
print("network ok")

def run(*cmd):
    """Run a step and let a failure actually stop the notebook."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)


In [ ]:
# 2. Get the Lilly code
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert Path("/kaggle/working/Lilly/training").is_dir(), "clone produced nothing"
os.chdir("/kaggle/working/Lilly")
print("working in", os.getcwd())


In [ ]:
# 3. Install what we need (~2 min) — same versions as on the Mac
run(sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.49.0", "peft==0.14.0", "accelerate==1.3.0",
    "sacrebleu", "sentencepiece", "sacremoses")


In [ ]:
# 4. Find the base weights among the datasets you attached
import glob
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/source.spm", recursive=True)]
assert found, ("Attach your models/lilly/translate folder as a Dataset input "
               "(right panel -> Input -> Add Input -> Datasets)")
assert os.path.exists(os.path.join(found[0], "config.json")), f"{found[0]} looks incomplete"
os.environ["LILLY_BASE"] = found[0]
print("base weights:", found[0])


In [ ]:
# 5. Download and clean the Bosnian-English data (~5 min)
run("python3", "data/scripts/download_data.py")
run("python3", "data/scripts/clean_data.py")

# The downloader reports a failed corpus and carries on, so check the result rather
# than the exit code. STATS.md records 334,790 training pairs; anything near that is fine.
pairs = sum(1 for _ in open("data/clean/train.tsv", encoding="utf-8"))
assert pairs > 200_000, f"only {pairs:,} training pairs — a corpus failed to download"
print(f"{pairs:,} training pairs ready")


In [ ]:
# 6. Quick pipeline check (~3 min) — a toy run, just to prove everything works.
# It writes to models/quicktest-adapter, never to the real one.
run("python3", "training/train_translation.py", "--quick-test")


In [ ]:
# 7. THE REAL TRAINING (~2-3 hours)
run("python3", "training/train_translation.py")
assert Path("models/lilly/adapter/adapter_config.json").is_file(), "no adapter was written"


In [ ]:
# 8. Score it: untuned base vs our Lilly, on all 1,500 sentences it never saw (~40 min)
# One run produces both rows. This table is the claim the whole project rests on,
# so it is measured on the entire held-out set, not a sample of it.
run("python3", "training/evaluate.py", "--adapter", "models/lilly/adapter")
print(Path("training/RESULTS.md").read_text())


In [ ]:
# 9. Package the result so it survives the run
assert Path("training/RESULTS.md").is_file()
run("zip", "-qr", "/kaggle/working/lilly-adapter.zip",
    "models/lilly/adapter", "training/RESULTS.md")
size = Path("/kaggle/working/lilly-adapter.zip").stat().st_size
assert size > 1_000_000, f"the zip is only {size} bytes"
print(f"lilly-adapter.zip — {size / 1048576:.1f} MB, in the Output tab when this finishes")


**Done.** Download `lilly-adapter.zip` from this version's **Output** tab and unzip it
so the adapter sits at `models/lilly/adapter/`. The app uses it the next time it starts.

Read `RESULTS.md` first. If the tuned row is not above the base row, the run did not help
and there is nothing worth shipping — more epochs or more data before another attempt.
